# Tutorial: Using the Vertex AI Gemini Generator Module

This notebook demonstrates how to use the `vertex.py` module, a convenient wrapper for interacting with the Google Cloud Vertex AI Gemini API. This module simplifies content generation by handling client initialization and providing a streamlined interface for various generation parameters, including structured output.

**Key Features:**
-   Easy initialization with `project_id` and `location`.
-   Support for different Gemini models (`GeminiModels.GEMINI_2_5_PRO`, `GeminiModels.GEMINI_2_5_FLASH`).
-   Content generation with text prompts.
-   Customizable generation parameters like `temperature`, `top_k`, `top_p`.
-   Ability to provide `system_instruction` for guiding model behavior.
-   Support for structured JSON output with a defined `output_schema`.

## 1. Setup and Installation

First, we need to install the `google-generativeai` library, which is the underlying SDK for interacting with Gemini models, and `strenum` for the `StrEnum` functionality used in the module.

In [ ]:
!pip install -q google-generativeai strenum
print("Libraries installed successfully!")

## 2. The `vertex.py` Module Source Code

For demonstration purposes, we'll include the entire `vertex.py` module directly in this notebook. This makes the example self-contained.

**Important Note on Dependencies**: The original `vertex.py` references `conidk.core.base.Auth` and `conidk.core.base.Config`. For this self-contained notebook, we define simple dummy `Auth` and `Config` classes. In a real application, you would ensure `conidk` is installed and imported correctly.

**Important Note on API Compatibility**: The original `vertex.py` passes `safety_settings` within `config` to `generate_content`. However, for the public `google-generativeai` client, `safety_settings` is a top-level argument to `generate_content`, and the generation parameters are passed via `generation_config`. The code below has been adjusted for compatibility with the latest `google-generativeai` client.

In [ ]:
# Dummy classes for `conidk.core.base` for demonstration purposes.
# In a real application, these would be properly imported.
class Auth:
    pass

class Config:
    pass

# --- Start of vertex.py content ---
from typing import Any, Dict, Optional
from strenum import StrEnum
from google.generativeai import types
import google.generativeai as genai


class GeminiModels(StrEnum):
    """Enum for supported Gemini models."""

    GEMINI_2_5_PRO = "gemini-2.5-pro"
    GEMINI_2_5_FLASH = "gemini-2.5-flash"


class MimeTypes(StrEnum):
    """Enum for supported MIME types."""

    TEXT_PLAIN = "text/plain"
    APPLICATION_JSON = "application/json"


_DEFAULT_SAFETY_SETTINGS = [
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_HARASSMENT,
        threshold=types.HarmBlockThreshold.BLOCK_NONE,
    ),
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH,
        threshold=types.HarmBlockThreshold.BLOCK_NONE,
    ),
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT,
        threshold=types.HarmBlockThreshold.BLOCK_NONE,
    ),
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT,
        threshold=types.HarmBlockThreshold.BLOCK_NONE,
    ),
    types.SafetySetting(
        category=types.HarmCategory.HARM_CATEGORY_CIVIC_INTEGRITY,
        threshold=types.HarmBlockThreshold.BLOCK_NONE,
    ),
]


class Generator:
    """A class to generate content using the Gemini models on Google Cloud Vertex AI.

    This class provides a simple, reusable interface for interacting with the Gemini API,
    handling client initialization and content generation with configurable parameters.

    Attributes:
        project_id: The Google Cloud project ID.
        location: The location where the Vertex AI endpoint is hosted (e.g., 'us-central1').
        version: The name of the Gemini model to use.
        auth: An authentication object.
        config: A configuration object.
        client: A Gemini API client.
    """

    def __init__(
        self,
        project_id: str,
        location: str,
        version: Optional[GeminiModels] = None,
        auth: Optional[Auth] = None, # Using the dummy Auth
        config: Optional[Config] = None, # Using the dummy Config
    ):
        """Initializes the Generator with project and location details.

        Args:
            project_id: The Google Cloud project ID.
            location: The location where the Vertex AI endpoint is hosted.
            version: The name of the Gemini model to use.
            auth: An authentication object.
            config: A configuration object.
        """
        self.auth = auth or Auth()
        self.config = config or Config()

        self.project_id = project_id
        self.location = location
        self.version = version if version is not None else GeminiModels.GEMINI_2_5_FLASH

        # Initialize the Vertex AI client
        self.client = genai.Client(
            vertexai=True, project=self.project_id, location=self.location
        )

    def content(
        self,
        prompt: str,
        system_instruction: Optional[str] = None,
        top_k: Optional[int] = 40,
        temperature: Optional[float] = 1,
        top_p: Optional[float] = 0.95,
        output_schema: Optional[Dict[str, Any]] = None,
        output_mime_type: str = MimeTypes.TEXT_PLAIN,
    ) -> str:
        """Generates content based on a text prompt using the configured Gemini model.

        Args:
            prompt: The user's input prompt as a string.
            system_instruction: An optional system instruction to guide the model's behavior.
            top_k: The number of highest probability tokens to consider.
            temperature: A value controlling the randomness of the output.
            top_p: The cumulative probability of tokens to consider.
            output_mime_type: The desired MIME type of the output.
            output_schema: An optional schema to enforce a specific structure on the output.

        Returns:
            The generated content as a string.
        """

        raw_gemini_response = self.client.models.generate_content(
            model=self.version,
            contents=[prompt],
            generation_config=types.GenerationConfig( # Use generation_config keyword for parameters
                system_instruction=system_instruction,
                candidate_count=1,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
                response_mime_type=output_mime_type,
                response_schema=output_schema,
            ),
            safety_settings=_DEFAULT_SAFETY_SETTINGS, # safety_settings is a top-level argument
        )
        if (
            raw_gemini_response.candidates
            and raw_gemini_response.candidates[0].content
            and raw_gemini_response.candidates[0].content.parts
        ):
            return raw_gemini_response.candidates[0].content.parts[0].text or ""
        return ""
# --- End of vertex.py content ---

print("Generator, GeminiModels, and MimeTypes classes loaded from vertex.py!")

## 3. Google Cloud Project Configuration and Authentication

To use Vertex AI, you need a Google Cloud project and appropriate authentication. Replace the placeholders below with your actual Google Cloud Project ID and the desired location for the Vertex AI endpoint.

**Authentication Steps (Choose one):**

1.  **Recommended for Colab:** Run `gcloud auth application-default login` in your local terminal and then restart the Colab runtime.
2.  **Service Account:** Set the `GOOGLE_APPLICATION_CREDENTIALS` environment variable to the path of your service account key JSON file.
3.  **Colab Secrets:** In a Colab environment, you can use the 'Secrets' tab (key icon on the left panel) to store your `GOOGLE_APPLICATION_CREDENTIALS` content or `PROJECT_ID` directly, and then load it into environment variables.

For this tutorial, we assume you have authenticated your environment.

In [ ]:
# Replace with your Google Cloud Project ID and desired location
PROJECT_ID = "YOUR_PROJECT_ID" # e.g., "my-gemini-project-12345"
LOCATION = "us-central1" # e.g., "us-central1", "europe-west4"

if PROJECT_ID == "YOUR_PROJECT_ID":
    raise ValueError("Please replace 'YOUR_PROJECT_ID' with your actual Google Cloud Project ID and run this cell again.")

print(f"Project ID: {PROJECT_ID}")
print(f"Location: {LOCATION}")

## 4. Initializing the `Generator`

Now we can create an instance of the `Generator` class, passing our project ID and location. We can also explicitly select a Gemini model version using the `GeminiModels` enum, or let it default to `GEMINI_2_5_FLASH`.

In [ ]:
# Initialize the generator with the default model (GEMINI_2_5_FLASH)
generator_default = Generator(project_id=PROJECT_ID, location=LOCATION)
print(f"Generator initialized with default model: {generator_default.version}")

# Initialize the generator with a specific model
generator_pro = Generator(
    project_id=PROJECT_ID,
    location=LOCATION,
    version=GeminiModels.GEMINI_2_5_PRO
)
print(f"Generator initialized with specific model: {generator_pro.version}")

## 5. Basic Text Generation

Let's start by generating some simple text content using a basic prompt.

In [ ]:
prompt = "Tell me a short, imaginative story about a robot who discovers a lost alien artifact."
print(f"Prompt:\n{prompt}\n")

response = generator_default.content(prompt=prompt)
print(f"Generated Story:\n---\n{response}\n---")

## 6. Using System Instructions

System instructions allow you to guide the model's overall behavior or persona. This is useful for making the model act as a specific character or follow certain rules.

In [ ]:
prompt = "What is the capital of France?"
system_instruction = "You are a helpful assistant who always answers in a poetic, rhyming style."
print(f"Prompt: {prompt}")
print(f"System Instruction: {system_instruction}\n")

response = generator_default.content(prompt=prompt, system_instruction=system_instruction)
print(f"Poetic Answer:\n---\n{response}\n---")

## 7. Adjusting Generation Parameters (`temperature`, `top_k`, `top_p`)

You can control the creativity and diversity of the generated content by adjusting parameters like `temperature`, `top_k`, and `top_p`.

-   **`temperature`**: Controls the randomness of predictions. Lower values produce more deterministic output, higher values more creative.
-   **`top_k`**: The number of top-k most likely tokens to sample from.
-   **`top_p`**: The cumulative probability threshold for token sampling.

In [ ]:
prompt = "Write a few sentences about the future of space exploration."
print(f"Prompt:\n{prompt}\n")

print("--- Low Temperature (more deterministic) ---")
response_low_temp = generator_default.content(
    prompt=prompt,
    temperature=0.2
)
print(f"Response 1 (temp=0.2):\n{response_low_temp}\n")

print("--- High Temperature (more creative) ---")
response_high_temp = generator_default.content(
    prompt=prompt,
    temperature=1.5,
    top_k=20,
    top_p=0.8
)
print(f"Response 2 (temp=1.5, top_k=20, top_p=0.8):\n{response_high_temp}\n")

## 8. Structured Output (JSON)

One of the powerful features is the ability to request structured output, such as JSON, by specifying `output_mime_type` and `output_schema`. This helps ensure the model's response adheres to a predefined format, making it easier to parse programmatically.

We'll need the `json` library to parse the output.

In [ ]:
import json

prompt = "Extract the title, author, and a short summary from the following text: 'The Hitchhiker's Guide to the Galaxy by Douglas Adams is a comedic science fiction series. It follows the adventures of the last surviving man, Arthur Dent, after the Earth is destroyed to make way for a hyperspace bypass.'"

output_schema = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "author": {"type": "string"},
        "summary": {"type": "string"}
    },
    "required": ["title", "author", "summary"]
}

print(f"Prompt:\n{prompt}\n")
print(f"Desired Output Schema:\n{json.dumps(output_schema, indent=2)}\n")

json_response_str = generator_default.content(
    prompt=prompt,
    output_mime_type=MimeTypes.APPLICATION_JSON,
    output_schema=output_schema
)

print(f"Raw JSON Response String:\n---\n{json_response_str}\n---")

try:
    parsed_json = json.loads(json_response_str)
    print("\nParsed JSON Output:")
    print(json.dumps(parsed_json, indent=2))
    print(f"\nTitle: {parsed_json['title']}")
    print(f"Author: {parsed_json['author']}")
except json.JSONDecodeError as e:
    print(f"Error parsing JSON: {e}")
    print("Model might not have returned valid JSON according to schema.")
except KeyError as e:
    print(f"Error accessing key in JSON: {e}")
    print("Model might not have returned all required fields.")

## Conclusion

This tutorial provided a comprehensive overview of the `vertex.py` module, demonstrating how to initialize the `Generator`, perform basic text generation, use system instructions, control generation parameters, and leverage structured JSON output. This wrapper simplifies common interactions with the Vertex AI Gemini API, making it easier to integrate powerful large language models into your applications.